<a href="https://colab.research.google.com/github/yaranbarzi/AiNotebook/blob/main/SRT_Splitt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 📂 آپلود زیرنویس
from google.colab import files

SRT_FILE = None
READY = False

print("📄 فایل زیرنویس (.srt) را انتخاب کنید:")
for f in files.upload():
    if f.lower().endswith('.srt'):
        SRT_FILE = f
        break

if SRT_FILE:
    READY = True
    print(f"✅ آماده شد: {SRT_FILE}")
    print("▶️ حالا سلول تنظیمات را اجرا کنید.")
else:
    print("❌ فایل .srt پیدا نشد. این سلول را دوباره اجرا کنید.")


In [ ]:
#@title 🎯 تنظیمات و ساخت زیرنویس
Words_Per_Line = ""      #@param {type:"string"}
Balance_Lines = True      #@param {type:"boolean"}
# Words_Per_Line: حداکثر تعداد کلمه در هر دیالوگ. خالی بگذاری ۵ حساب می‌شود.
#                 ۰ یعنی هر دیالوگ همان‌طور که هست بماند.
#                 اگر ترجمه‌ی فارسی از جمله‌ی اصلی بلندتر است، عدد را بزرگ‌تر کن (مثلاً ۷ یا ۸) تا دیالوگ‌ها خیلی کوتاه نشوند.
# Balance_Lines: وقتی جمله شکسته می‌شود، تکه‌ها هم‌اندازه شوند (برای جمله‌ی کوتاه‌تر از حداکثر اثری ندارد).

import re, math, os
from google.colab import files
from IPython.display import display, HTML

HOLD_SECONDS = 0.15   # زیرنویس بعد از تمام‌شدن حرف این‌قدر بیشتر می‌ماند
MIN_DURATION = 0.9    # کمترین مدت نمایش هر زیرنویس

def _fmt(sec):
    ms = int(round(max(sec, 0) * 1000))
    h, r = divmod(ms, 3600000); m, r = divmod(r, 60000); s, ms = divmod(r, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

def _read_srt(path):
    raw = open(path, "rb").read()
    for enc in ("utf-8-sig", "utf-16", "cp1256", "latin-1"):
        try: txt = raw.decode(enc); break
        except Exception: continue
    txt = txt.replace("\r\n", "\n").replace("\r", "\n").strip()
    out = []
    for b in re.split(r"\n\s*\n", txt):
        lines = b.split("\n")
        i = next((k for k, l in enumerate(lines) if "-->" in l), None)
        if i is None: continue
        m = re.match(r"\s*(\d+):(\d+):(\d+)[,.](\d+)\s*-->\s*(\d+):(\d+):(\d+)[,.](\d+)", lines[i])
        if not m: continue
        g = m.groups()
        tt = lambda h, mi, s, ms: int(h) * 3600 + int(mi) * 60 + int(s) + float("0." + ms)
        text = re.sub(r"<[^>]+>|\{[^}]+\}", "", " ".join(lines[i + 1:]))
        text = re.sub(r"\s+", " ", text).strip()
        if text: out.append({"s": tt(*g[:4]), "e": tt(*g[4:]), "t": text})
    return out

def _sizes(m, n, balance):
    if n <= 0 or m <= n: return [m]
    if balance:
        k = math.ceil(m / n); base, extra = divmod(m, k)
        return [base + 1 if i < extra else base for i in range(k)]
    sz = [n] * (m // n)
    if m % n: sz.append(m % n)
    return sz

def _spread(cues):
    """زمان شروع و پایان هر دیالوگ همان می‌ماند و بین کلمه‌ها به نسبت طولشان پخش می‌شود."""
    out = []
    for c in cues:
        ws = c["t"].split()
        tot = sum(len(x) + 1 for x in ws)
        span = max(c["e"] - c["s"], 0.05 * len(ws))
        t0, w = c["s"], []
        for x in ws:
            d = span * (len(x) + 1) / tot
            w.append([x, t0, t0 + d]); t0 += d
        out.append(w)
    return out

def show_subscribe_banner():
    image_url = 'https://huggingface.co/Toolsai/dubtest/resolve/main/newgolden.png'
    youtube_channel_url = 'https://youtube.com/@aigolden'
    display(HTML(f"""
    <div style="text-align: center; border: 2px solid #e0e0e0; padding: 15px; border-radius: 12px; background-color: #f9f9f9; max-width: 350px; margin: auto;">
        <a href="{youtube_channel_url}" target="_blank" title="رفتن به کانال یوتیوب AIGOLDEN">
            <img src="{image_url}" alt="AIGOLDEN YouTube Channel" style="max-width: 100%; height: auto; border-radius: 8px;">
        </a>
        <p style="font-size: 16px; font-family: 'Vazir', sans-serif; margin-top: 15px; color: #333;">
            برای مشاهده آموزش‌های بیشتر، ما را در یوتیوب دنبال کنید.
        </p>
        <a href="{youtube_channel_url}" target="_blank" style="text-decoration: none; display: inline-block; background-color: #FF0000; color: white; padding: 10px 20px; border-radius: 8px; font-weight: bold; font-family: 'Vazir', sans-serif; margin-top: 10px;">
            🚀 دنبال کردن در یوتیوب
        </a>
    </div>"""))

# ---------- اجرا ----------
if not globals().get("READY"):
    print("❌ ابتدا سلول آپلود را اجرا کنید.")
else:
    try:
        try: N = int(str(Words_Per_Line).strip() or 5)
        except Exception: N = 5
        if N < 0: N = 5

        cues = _read_srt(SRT_FILE)
        assert cues, "هیچ دیالوگی در فایل SRT پیدا نشد."
        per_cue = _spread(cues)

        prev = 0.0                       # زمان‌ها باید صعودی باشند
        for w in per_cue:
            for x in w:
                x[1] = max(x[1], prev); x[2] = max(x[2], x[1] + 0.02); prev = x[1]

        out = []
        for w in per_cue:
            i = 0
            for sz in _sizes(len(w), N, bool(Balance_Lines)):
                g = w[i:i + sz]; i += sz
                out.append({"s": g[0][1], "e": g[-1][2], "t": " ".join(x[0] for x in g)})
        for i, c in enumerate(out):
            nxt = out[i + 1]["s"] if i + 1 < len(out) else 1e9
            e = min(c["e"] + HOLD_SECONDS, nxt)
            if e - c["s"] < MIN_DURATION: e = min(c["s"] + MIN_DURATION, nxt)
            c["e"] = max(e, c["s"] + 0.05)

        OUT_FILE = os.path.splitext(SRT_FILE)[0] + "_split.srt"
        with open(OUT_FILE, "w", encoding="utf-8") as f:
            for i, c in enumerate(out, 1):
                f.write(f"{i}\n{_fmt(c['s'])} --> {_fmt(c['e'])}\n{c['t']}\n\n")

        print(f"✅ از {len(cues)} دیالوگ، {len(out)} دیالوگ ساخته شد → {OUT_FILE}")
        short = sum(1 for c in out if c["e"] - c["s"] < 0.8)
        if short > len(out) * 0.2:
            print(f"⚠️ {short} دیالوگ کمتر از ۰.۸ ثانیه نمایش داده می‌شوند. برای خوانایی بهتر Words_Per_Line را بزرگ‌تر کنید.")
        print("\n▶️ نمونه:")
        for c in out[:5]:
            print(f"   {_fmt(c['s'])} → {_fmt(c['e'])}   {c['t']}")

        print(f"\n📥 در حال دانلود: {OUT_FILE}")
        files.download(OUT_FILE)
        show_subscribe_banner()
    except Exception as e:
        print("⚠️ ساخت زیرنویس ناموفق بود.")
        print("   علت:", repr(e))
